### **Midterm Analysis Code | Tyler Hilbert | April 28, 2026**

The purpose of this code is to generate the dataframes needed to create figures that analyze *Midterm Grade Data*. The file will be notated with instructions for future reference and areas of improvement.

*Note: This is my first pass at converting my prior code to be prepared for the final project. However, I realized after I translated most of it that I'd need to redo this to do some coding work for the other figures (namely the line charts and heat map) - I continued with it to focus on generating HTML code. To see the really new stuff, jump to **Section 2** and **Section 5** - most of the table code was adjusted to utilize loops/fit what is new in those. The other coding blocks were kept the same/cutting out the fluff (failed attempts, experimental code, etc.)*

#### **Section 0 - Importing Libraries**

All libraries used for the code are imported here, along with purpose. Please ensure all libraries are loaded into the kernel prior to running the code. For library installation methods, please refer to the *Library Installation Guide.ipynb* file.


In [1]:
import pandas as pd #allows for data manipulation
import numpy as np #allows for additional logic (i.e. reading lists, if statements)
import plotly.graph_objects as go #allows for creation of figures and visualizations
import plotly.io as pio #allows for conversion of plotly code to HTML file for uploading to PowerBI
import json #Allows figures to be converted to JSON files for posting onto PowerBI

#### **Section 1 - Dataframes**

The code in this section is dedicated to creating the dataframes that will be used for visualizations. By the end of the section, three frames will be created:
- A dataframe of midterm grades for all enrolled students
- A dataframe limited to students in the CAED-CCI-CotA hub
- A dataframe limited to students in the CAED-CCI-CotA hub that require advisor intervention

**Please Note:** The code in this file is focused on creating the dataframe only - code for reading the dataframe (i.e. checking shape, size, keys) are not included. Additional sizing code should be conducted separately to maintain the integrity of this file.

In [2]:
#Section 1a - Loading Data
#Purpose - loads data into the script for manipulation
mtfull = pd.read_csv("mtgradesanon.csv") #Ensure that the name of the file aligns with the CSV file you are pulling
mtfull.head()

,Unnamed: 0,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class
0,0,RW,AED,22860,KC,A-,B,ARCH,ID,202280,FR
1,1,RW,AED,22860,KC,A,A,ARCH,ID,202280,FR
2,2,RW,AED,22860,KC,A,A,ARCH,OTH,202280,JR
3,3,RW,AED,22860,KC,F,B,ARCH,ID,202280,SO
4,4,RW,AED,22860,KC,B,B,ARCH,ID,202280,FR


In [3]:
#Section 1b - Renaming Column Header
#Purpose - Renames index column to something usable later
mtfull = mtfull.rename(columns = {"Unnamed: 0":"Record ID"})
mtfull.head()

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class
0,0,RW,AED,22860,KC,A-,B,ARCH,ID,202280,FR
1,1,RW,AED,22860,KC,A,A,ARCH,ID,202280,FR
2,2,RW,AED,22860,KC,A,A,ARCH,OTH,202280,JR
3,3,RW,AED,22860,KC,F,B,ARCH,ID,202280,SO
4,4,RW,AED,22860,KC,B,B,ARCH,ID,202280,FR


##### *Creating a Course Code Column*

In [4]:
#Section 1c - Prepping Course to Become STR
#Purpose - allows for Course Code column to be made (Course + Number)
mtfull["Course"] = mtfull["Course"].astype(str)

In [5]:
#Section 1d - Creating Course Code Column
#Purpose - Creating an item for future filtering
mtfull["Course Code"] = mtfull["Subject"] + " " + mtfull["Course"]
mtfull.head()

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code
0,0,RW,AED,22860,KC,A-,B,ARCH,ID,202280,FR,AED 22860
1,1,RW,AED,22860,KC,A,A,ARCH,ID,202280,FR,AED 22860
2,2,RW,AED,22860,KC,A,A,ARCH,OTH,202280,JR,AED 22860
3,3,RW,AED,22860,KC,F,B,ARCH,ID,202280,SO,AED 22860
4,4,RW,AED,22860,KC,B,B,ARCH,ID,202280,FR,AED 22860


##### *Translating Registration Status Codes*

In [6]:
#Section 1e - Pulling Reg Codes
#Purpose - Identify all reg codes in the file
#Note that there may be new codes - if so, review Banner documentation to identify translation
regstatuscodes = mtfull["Registration Status"].unique()
regstatuscodes

<StringArray>
['RW', 'RE', 'WW', 'DD', 'W8', 'ND', 'R2', 'SF', 'B1', 'WD', 'NF', 'RA', 'AW',
 'DR', 'B5']
Length: 15, dtype: str

In [7]:
#Section 1f - Writing Code Translation
#Purpose - Create a list of translations that match the codes above
#The list below matches the list created above - if list gets reordered, rewrite this code accordingly.
regstatuscodestranslated = ["Registered","Std Withdrawn","Stopped Attending - Failed", "Admin Dropped","Std Dropped","Admin Dropped","Never Attended - Failed", "Registered", "Std Withdrawn", "Registered", "Admin Withdrawn", "Audited", "Admin Dropped", "Admin Withdrawn", "Std Dropped"]
regstatuscodestranslated

['Registered',
 'Std Withdrawn',
 'Stopped Attending - Failed',
 'Admin Dropped',
 'Std Dropped',
 'Admin Dropped',
 'Never Attended - Failed',
 'Registered',
 'Std Withdrawn',
 'Registered',
 'Admin Withdrawn',
 'Audited',
 'Admin Dropped',
 'Admin Withdrawn',
 'Std Dropped']

In [8]:
#Section 1g - Mapping the Code
#Purpose - Creating a map so we can replace the codes with the full translation
mapregstatuscode = dict(zip(regstatuscodes, regstatuscodestranslated))
mapregstatuscode

{'RW': 'Registered',
 'RE': 'Std Withdrawn',
 'WW': 'Stopped Attending - Failed',
 'DD': 'Admin Dropped',
 'W8': 'Std Dropped',
 'ND': 'Admin Dropped',
 'R2': 'Never Attended - Failed',
 'SF': 'Registered',
 'B1': 'Std Withdrawn',
 'WD': 'Registered',
 'NF': 'Admin Withdrawn',
 'RA': 'Audited',
 'AW': 'Admin Dropped',
 'DR': 'Admin Withdrawn',
 'B5': 'Std Dropped'}

In [9]:
#Section 1h - Replacing the Reg Codes w/ Mapping
#Purpose - Replace the codes w/ the translations via mapping
mtfull["Registration Status"] = mtfull["Registration Status"].map(mapregstatuscode)
mtfull.head()

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,202280,FR,AED 22860
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,202280,FR,AED 22860
2,2,Registered,AED,22860,KC,A,A,ARCH,OTH,202280,JR,AED 22860
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,202280,SO,AED 22860
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,202280,FR,AED 22860


##### *Creating a Major College Column*

In [10]:
#Section 1i - Identifying Majors
#Purpose - To pull a list of majors in the file
mtfull["Major"].unique()

<StringArray>
[  'ID',  'OTH', 'ARCH',   'FM', 'COMM', 'ARTH',   'FD', 'DNST', 'TDTP',
 'SART', 'COMA',  'DMP',  'VCD',  'JNL', 'EMAT',  'ADV', 'THEA', 'ARCS',
   'PR', 'ARTE', 'MUST', 'PHOT',  'MUS', 'MUED',  'MUT', 'APMD', 'UXDE',
 'DANC']
Length: 28, dtype: str

In [11]:
#Section 1j - Assigning Majors to Colleges
#Purpose - create variables that capture all majors in each college.
#Note that this file is concerned with CAED, CCI and CotA only. All other majors will be coded as OTH college.
caedmajor = ["ID", "ARCH", "COMA", "ARCS"] #Majors in CAED
ccimajor = ["COMM", "DMP", "VCD", "JNL", "EMAT", "ADV", "PR", "PHOT", "APMD","UXDE"] #Majors in CCI
cotamajor = ["FM", "ARTH", "FD", "DNST", "TDTP", "SART", "THEA", "ARTE", "MUST", "MUS", "MUED", "MUT","DANC"] #Majors in CotA
othermajor = ["OTH"]

In [12]:
#Section 1k - Major -> College Conditions & Outcomes
#Purpose - create conditions and outcomes that will check if majors are in one of the affected colleges
majcond = [
    mtfull["Major"].isin(caedmajor),
    mtfull["Major"].isin(ccimajor),
    mtfull["Major"].isin(cotamajor),
    mtfull["Major"].isin(othermajor)] 

majout = ["CAED", "CCI", "CotA", "Other"]

In [13]:
#Section 1l - Plugging Conditions/Outcomes and Creation of Major College
#Purpose - Using Conditions/Outcomes to fill in a Major College column
mtfull["Major College"] = np.select(majcond,majout, "Missing") #Use Missing Value to catch any missed majors
mtfull["Major College"].unique() #Run to check for missing majors

<StringArray>
['CAED', 'Other', 'CotA', 'CCI']
Length: 4, dtype: str

##### *Creating a Subject College Column*

In [14]:
#Section 1m - Identifying Subjects
#Purpose - Create a list of Subjects that are examined this semester
mtfull["Subject"].unique()

<StringArray>
[ 'AED', 'ARCH', 'ARCS',  'ART', 'ARTH', 'ARTS',  'CCI', 'CMGT', 'COMM',
  'DAN', 'EMAT',  'FDM',   'ID',  'MDJ',  'MUS', 'THEA',  'VCD']
Length: 17, dtype: str

In [15]:
#Section 1n - Making Subject Lists
#Purpose - Create variables that capture all subjects and aligns to a college
caedsubject = ["AED", "ARCH", "ARCS", "CMGT", "ID"] #Subjects in CAED
ccisubject = ["CCI", "COMM", "EMAT", "MDJ", "VCD"] #Subjects in CCI
cotasubject = ["ART", "ARTH", "ARTS", "DAN", "FDM", "MUS", "THEA"] #Subjects in CotA

In [16]:
#Section 1o - Subject -> College Outcomes & Condition
#Purpose - Create the conditions and outcomes that check if the subject belongs in one of the listed college
subcond = [
    mtfull["Subject"].isin(caedsubject),
    mtfull["Subject"].isin(ccisubject),
    mtfull["Subject"].isin(cotasubject)] 

subout = ["CAED", "CCI", "CotA"]

In [17]:
#Section 1p - Plugging Conditions/Outcome to Make Subject College column
#Purpose - Using conditions/outcomes to fill in a subject college column
mtfull["Subject College"] = np.select(subcond,subout, "Missing") #Use Missing value to check for missed subjects
mtfull["Subject College"].unique() #Run to ensure no subjects were missed

<StringArray>
['CAED', 'CotA', 'CCI']
Length: 3, dtype: str

##### *Assigning W Midterm Grades*

In [18]:
#Section 1q - Withdrawal w/o MT Check
#Purpose - Checking if someone withdrew w/o a midterm grade (withdrew pre-MT season)
mtfull["Withdrew No MT"] = np.where((mtfull["Final Grade"] == "W") & (mtfull["Mid Term Grade"].isna()), True, False)
mtfull.head()

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College,Withdrew No MT
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,202280,FR,AED 22860,CAED,CAED,False
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,202280,FR,AED 22860,CAED,CAED,False
2,2,Registered,AED,22860,KC,A,A,ARCH,OTH,202280,JR,AED 22860,Other,CAED,False
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,202280,SO,AED 22860,CAED,CAED,False
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,202280,FR,AED 22860,CAED,CAED,False


In [19]:
#Section 1r - Withdrawal w/o MT W Entry
#Purpose - Filling Grade column with a W if they withdrew before a MT grade was entered
mtfull["Mid Term Grade"] = np.where(mtfull["Withdrew No MT"] == True, "W", mtfull["Mid Term Grade"])
mtfull["Mid Term Grade"].unique()

<StringArray>
[ 'B',  'A',  nan, 'A-',  'C', 'C+',  'D', 'B+', 'B-', 'C-',  'F',  'S', 'D+',
  'W',  'U', 'SF', 'NF']
Length: 17, dtype: str

In [20]:
#Section 1s - Removal of Withdrew No MT Column
#Purpose - Removing the column - not necessary for future analysis
mtfull = mtfull.drop("Withdrew No MT", axis=1)
mtfull.head()

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,202280,FR,AED 22860,CAED,CAED
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,202280,FR,AED 22860,CAED,CAED
2,2,Registered,AED,22860,KC,A,A,ARCH,OTH,202280,JR,AED 22860,Other,CAED
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,202280,SO,AED 22860,CAED,CAED
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,202280,FR,AED 22860,CAED,CAED


##### *Assigning Dropped Midterm Grades*

In [21]:
#Section 1t - Creating a Dropped Grade Conditional Column
#Purpose - To create a column that checks if the student dropped the course prior to MT grades.
mtfull["Dropped no MT"] = np.where(((mtfull["Registration Status"] == "Std Dropped") | (mtfull["Registration Status"] == "Admin Dropped")) & (mtfull["Mid Term Grade"].isna()), True, False)
mtfull.head()

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College,Dropped no MT
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,202280,FR,AED 22860,CAED,CAED,False
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,202280,FR,AED 22860,CAED,CAED,False
2,2,Registered,AED,22860,KC,A,A,ARCH,OTH,202280,JR,AED 22860,Other,CAED,False
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,202280,SO,AED 22860,CAED,CAED,False
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,202280,FR,AED 22860,CAED,CAED,False


In [22]:
#Section 1u - Plugging in the DR grades
#Purpose - To put a DR grade in for students who dropped from the course prior to MT
mtfull["Mid Term Grade"] = np.where(mtfull["Dropped no MT"] == True, "DR", mtfull["Mid Term Grade"])
mtfull["Mid Term Grade"].unique()

<StringArray>
[ 'B',  'A',  nan, 'A-',  'C', 'C+',  'D', 'B+', 'B-', 'C-',  'F',  'S', 'D+',
 'DR',  'W',  'U', 'SF', 'NF']
Length: 18, dtype: str

In [23]:
#Section 1v - Removal of the Dropped no MT column
#Purpose - Removing the column - not necessary for future analysis
mtfull = mtfull.drop("Dropped no MT", axis=1)
mtfull.head()

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,202280,FR,AED 22860,CAED,CAED
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,202280,FR,AED 22860,CAED,CAED
2,2,Registered,AED,22860,KC,A,A,ARCH,OTH,202280,JR,AED 22860,Other,CAED
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,202280,SO,AED 22860,CAED,CAED
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,202280,FR,AED 22860,CAED,CAED


##### *Removal of X Final Grades*

In [24]:
#Section 1w - Removing X grades
#Purpose - Do to past terms being reviewed, need to remove X grades from record to accurately capture grade earned.
mtfull["Final Grade"] = mtfull["Final Grade"].str.lstrip("X")
mtfull["Final Grade"].unique()

<StringArray>
['A-',  'A',  'F',  'B',  nan, 'D+',  'C', 'B-', 'B+', 'C-',  'W', 'C+',  'S',
  'D', 'SF', 'NF', 'AU', 'IN', 'NR',  'U']
Length: 20, dtype: str

##### *Assigning Dropped Final Grades*

In [25]:
#Section 1x - Creating a Conditional column for Dropped Final Grades
#Purpose - To add a DR grade when a student dropped prior to a final grade being assigned - also accounts for Admin drops
mtfull["Dropped no Fin"] = np.where(((mtfull["Registration Status"] == "Std Dropped") | (mtfull["Registration Status"] == "Admin Dropped")) & (mtfull["Final Grade"].isna()), True, False)
mtfull.head()

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College,Dropped no Fin
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,202280,FR,AED 22860,CAED,CAED,False
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,202280,FR,AED 22860,CAED,CAED,False
2,2,Registered,AED,22860,KC,A,A,ARCH,OTH,202280,JR,AED 22860,Other,CAED,False
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,202280,SO,AED 22860,CAED,CAED,False
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,202280,FR,AED 22860,CAED,CAED,False


In [26]:
#Section 1y - Inputting a DR for Final Grades
#Purpose - Plugging in a Drop final grade for students who dropped from the course.
mtfull["Final Grade"] = np.where(mtfull["Dropped no Fin"] == True, "DR", mtfull["Final Grade"])
mtfull["Final Grade"].unique()

<StringArray>
['A-',  'A',  'F',  'B',  nan, 'D+',  'C', 'B-', 'B+', 'C-',  'W', 'C+',  'S',
  'D', 'DR', 'SF', 'NF', 'AU', 'IN', 'NR',  'U']
Length: 21, dtype: str

In [27]:
#Section 1z - Dropping the Dropped no Fin column
#Purpose - Removing the column - not necessary for future analysis
mtfull = mtfull.drop("Dropped no Fin", axis=1)
mtfull.head()

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,202280,FR,AED 22860,CAED,CAED
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,202280,FR,AED 22860,CAED,CAED
2,2,Registered,AED,22860,KC,A,A,ARCH,OTH,202280,JR,AED 22860,Other,CAED
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,202280,SO,AED 22860,CAED,CAED
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,202280,FR,AED 22860,CAED,CAED


##### *Assigning Numerical Value to MT and Final Grades*

The next step involves the creation of converting the grades to a numerical value. This conversion goes off of the university's GPA scale. There are some grades that do not correlate with a number due to not contributing toward GPA. Some of the grades assigned and logic can be found below:
- DR: will be a blank value. since they never earned a grade, there is no point in assigning a value since they would lead to a higher concentration of values in a certain spot.
- NR: will be an NaN. NR grades are assigned when the instructor never reported a grade.
- AU: will be a blank value - these are people auditing the course, so they never get a grade.
- S: will be a 4. S grades do not contribute to GPA, but they are a passing grade. Presumably there would be movement from S to U and vice versa, so U will be assigned a 0.
- U: will be a 0. See the logic for S
- W: will be a 0. W's do not contribute to GPA, but it is still a "negative" outcome for the course and is calculated in other retention efforts as a negative outcome (i.e. DFW rates are D, F and W grades).
- IN: will be a blank value. IN grades are put whenever students get an extension on entering their grades. It is interpreted similarly to W's (doesn't impact GPA), but since there is a chance for this to change, I don't want to count it (students could complete the work and come back with a passing grade)

*Grade Conversion Chart*
- A = 4.0
- A- = 3.7
- B+ = 3.3
- B = 3.0
- B- = 2.7
- C+ = 2.3
- C = 2.0
- C- = 1.7
- D+ = 1.3
- D = 1.0
- F = 0.0
- W = 0.0
- S = 4.0
- U = 0.0
- SF = 0.0
- NF =  0.0
- AU = NaN
- NaN = NaN
- DR =  NaN
- IN =  NaN
- NR = NaN

In [28]:
#Section 1aa - Creating the Conversion Map
#Purpose - Allows for the conversion from grade to number. If additional grades are needed, add to map.
gpamap = {"A":4.0, "A-":3.7, "B+":3.3, "B":3.0, "B-":2.7, "C+":2.3, "C":2.0, "C-":1.7, "D+":1.3,
          "D":1.0, "F":0.0, "W":0.0, "S":4.0, "U":0.0, "SF":0.0, "NF":0.0,
          "AU": "NaN", "DR":"NaN", "IN":"NaN", "NR":"NaN", "NaN":"NaN"}

In [29]:
#Section 1ab - Creating a MT and Final Grade Number Column
mtfull["Mid Term Grade Number"] = mtfull["Mid Term Grade"].map(gpamap)
mtfull["Final Grade Number"] = mtfull["Final Grade"].map(gpamap)
mtfull.head()

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College,Mid Term Grade Number,Final Grade Number
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,202280,FR,AED 22860,CAED,CAED,3.0,3.7
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,202280,FR,AED 22860,CAED,CAED,4.0,4.0
2,2,Registered,AED,22860,KC,A,A,ARCH,OTH,202280,JR,AED 22860,Other,CAED,4.0,4.0
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,202280,SO,AED 22860,CAED,CAED,3.0,0.0
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,202280,FR,AED 22860,CAED,CAED,3.0,3.0


In [30]:
#Section 1ac - Converting Numbers to Numerical Data
#Purpose - Need to convert to numerical to be used as continuous data later. 
mtfull["Final Grade Number"] = pd.to_numeric(mtfull["Final Grade Number"], errors = "coerce")
mtfull["Mid Term Grade Number"] = pd.to_numeric(mtfull["Mid Term Grade Number"], errors = "coerce")
mtfull.dtypes

Record ID                  int64
Registration Status          str
Subject                      str
Course                       str
Campus                       str
Final Grade                  str
Mid Term Grade               str
Department                   str
Major                        str
Academic Period            int64
Class                        str
Course Code                  str
Major College                str
Subject College              str
Mid Term Grade Number    float64
Final Grade Number       float64
dtype: object

##### *Creating a Full Data File*

In [31]:
#Section 1ad - Creating the Full Data File
#Purpose - To create a CSV file that captures all students grade data. Will be used in creation of MT Summary Table.
mtfull.to_csv("mtfullclean.csv")

##### *Removing Non-Hub Majors*

In [32]:
#Section 1ae - Removing OTH Majors
#Purpose - Allows for the removal of non-hub majors from the file. This allows for creation of hub only CSV files.
mthubonly = mtfull[mtfull["Major"] != "OTH"].copy()
mthubonly.head()

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College,Mid Term Grade Number,Final Grade Number
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,202280,FR,AED 22860,CAED,CAED,3.0,3.7
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,202280,FR,AED 22860,CAED,CAED,4.0,4.0
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,202280,SO,AED 22860,CAED,CAED,3.0,0.0
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,202280,FR,AED 22860,CAED,CAED,3.0,3.0
5,5,Registered,AED,22860,KC,B,NaN,ARCH,ID,202280,FR,AED 22860,CAED,CAED,NaN,3.0


##### *Adding Advisor Intervention Columns*

In [33]:
#Section 1af - Intervention Criteria
#Purpose - Creates the criteria for the intervention columns later. Note that there is a differentitation between pre and post-202610 due to changes in process.
pre202610frsopassgrade = ["A", "A-", "B+","B","B-","C+","S"]
pre202610frsofailgrade = ["C", "C-", "D+", "D", "F","U","SF"]
post202610frsopassgrade = ["A", "A-", "B+","B","B-","C+","C", "C-","S"]
post202610frsofailgrade = ["D+", "D", "F","U", "SF"]
jrsrpassgrade = ["A", "A-", "B+","B","B-","C+","C","C-","D+","D","S"]
jrsrfailgrade = ["F","U","SF"]
dropgrade = ["DR"]
wgrade = ["W"]
nfgrade = ["NF"]
frsocheck = ["FR", "SO"]
jrsrcheck = ["JR", "SR"]
pre202610terms = [202280, 202310, 202380, 202410, 202480, 202510, 202580]
post202610terms = [202610]

In [34]:
#Section 1ag - Intervention Conditions/Outcomes
#Purpose - Creates the logic for the conditions and what the outcome should be when reviewing the data.
intercond = [
    ((mthubonly["Mid Term Grade"].isin(pre202610frsopassgrade)) & (mthubonly["Class"].isin(frsocheck)) & (mthubonly["Academic Period"].isin(pre202610terms))),
    ((mthubonly["Mid Term Grade"].isin(pre202610frsofailgrade)) & (mthubonly["Class"].isin(frsocheck)) & (mthubonly["Academic Period"].isin(pre202610terms))),
    ((mthubonly["Mid Term Grade"].isin(post202610frsopassgrade)) & (mthubonly["Class"].isin(frsocheck))  & (mthubonly["Academic Period"].isin(post202610terms))),
    ((mthubonly["Mid Term Grade"].isin(post202610frsofailgrade)) & (mthubonly["Class"].isin(frsocheck))  & (mthubonly["Academic Period"].isin(post202610terms))),
    ((mthubonly["Mid Term Grade"].isin(jrsrpassgrade)) & (mthubonly["Class"].isin(jrsrcheck))),
    ((mthubonly["Mid Term Grade"].isin(jrsrfailgrade)) & (mthubonly["Class"].isin(jrsrcheck))),
    mthubonly["Mid Term Grade"].isin(dropgrade),
    mthubonly["Mid Term Grade"].isin(wgrade),
    mthubonly["Mid Term Grade"].isin(nfgrade)
]

interout = ["Passing MT Grade", "FR/SO Intervention Needed", "Passing MT Grade", "FR/SO Intervention Needed", "Passing MT Grade", "JR/SR Intervention Needed", "Dropped Course", "Withdrew From Course", "Never Attended Course"]

In [35]:
#Section 1ah - Applying Conditions/Outcomes to Make Intervention Column
#Purpose - Putting the logic in to create the intervention column. Logic will run through above conditions to dtermine what status should be entered
mthubonly["MT Intervention Needed"] = np.select(intercond,interout, "No MT Grade Entered") #Remembering that the last bit is the "Catch all" value, I chose to make it the No MT Grade Entered
mthubonly.head()

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College,Mid Term Grade Number,Final Grade Number,MT Intervention Needed
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,202280,FR,AED 22860,CAED,CAED,3.0,3.7,Passing MT Grade
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,202280,FR,AED 22860,CAED,CAED,4.0,4.0,Passing MT Grade
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,202280,SO,AED 22860,CAED,CAED,3.0,0.0,Passing MT Grade
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,202280,FR,AED 22860,CAED,CAED,3.0,3.0,Passing MT Grade
5,5,Registered,AED,22860,KC,B,NaN,ARCH,ID,202280,FR,AED 22860,CAED,CAED,NaN,3.0,No MT Grade Entered


##### *Quick Fix for GR Students in the Course*

In [36]:
#Section 1ai - Quick Fix for GR Students
#Purpose - To address the GR students in lower level courses, quickly applying that they are passing to MT grades.
mthubonly.loc[(mthubonly["MT Intervention Needed"] == "No MT Grade Entered") & (mthubonly["Mid Term Grade"].notna()),"MT Intervention Needed"] = "Passing MT Grade" 
mthubonly.groupby("MT Intervention Needed").count()["Record ID"]

MT Intervention Needed
Dropped Course                 927
FR/SO Intervention Needed     5460
JR/SR Intervention Needed      652
Never Attended Course           82
No MT Grade Entered           5692
Passing MT Grade             45026
Withdrew From Course           948
Name: Record ID, dtype: int64

##### *Creating the Complete Hub File*

In [37]:
#Section 1aj - Creating Hub Full File
#Purpose - To create a version of the file that looks at just students in the hub. Not currently used for figures, but may be in the future.
mthubonly.to_csv("mthubonlyclean.csv")

##### *Removing Non-Intervention Students*

In [38]:
#Section 1ak - Removing non-Interventions
#Purpose - This will remove all students not requiring interventions from the file.
mthubinterventions = mthubonly[((mthubonly["MT Intervention Needed"] == "FR/SO Intervention Needed") | (mthubonly["MT Intervention Needed"] == "JR/SR Intervention Needed"))].copy()
mthubinterventions.head()

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College,Mid Term Grade Number,Final Grade Number,MT Intervention Needed
11,11,Registered,AED,22860,KC,NaN,C,ARCH,ID,202280,FR,AED 22860,CAED,CAED,2.0,NaN,FR/SO Intervention Needed
14,14,Registered,AED,22860,KC,C,D,ARCH,ID,202280,FR,AED 22860,CAED,CAED,1.0,2.0,FR/SO Intervention Needed
19,19,Registered,AED,22860,KC,B-,C,ARCH,ID,202280,SO,AED 22860,CAED,CAED,2.0,2.7,FR/SO Intervention Needed
31,31,Stopped Attending - Failed,AED,22860,KC,W,D,ARCH,ID,202280,SO,AED 22860,CAED,CAED,1.0,0.0,FR/SO Intervention Needed
33,33,Registered,AED,22860,KC,B-,C-,ARCH,ID,202280,FR,AED 22860,CAED,CAED,1.7,2.7,FR/SO Intervention Needed


##### *Creating the Intervention File*

In [39]:
#Section 1al - Creating the Intervention File
#Purpose - Create a file that includes onlt the students who require interventions. This is what will be turned into the intervention contact list for advisors.
mthubinterventions.to_csv("mthubinterventions.csv")

#### **Section 2 - MT Grade Summary Tables**

The code in the below section is focused on creating multiple tables that summarize the MT grade data by college. The purpose of this table is to provide college leadership (Dean and School Directors), an overview of how well students are doing in midterms. Through this, leadership can understand pain points for students, perform outreach to instructors, and guide discussion for curricular revisions. 

Note that this table looks at **all** enrolled students - focus is not placed on hub-only majors, as a majority of these courses can be taken by any student. As such, it will be referencing the **mtfull** data as a base to be manipulated.

Additionally, the tables are cut down to be viewed by college only, with filters to view by department (CotA and CCI) or subject (CAED). This is due to multiple subjects being within CotA and CCI departments, while CAED does not have individual departments, just subjects.

##### *Manipulating Dataframes For Summary Tables*

In [40]:
#Section 2a - Removing Dropped Students
#Purpose - Dropped grades are assigned prior to midterm - they cannot be used as a determinator of student success in a course. As such, they are removed from calculations.
mtfullnodrop = mtfull[mtfull["Mid Term Grade"] != "DR"].copy()
mtfullnodrop.shape

(83989, 16)

In [41]:
#Section 2b - Defining Pass v. Fail Grades
#Purpose - To create variables that can be referenced as pass/fail. What is determined as pass/fail remains the same, no matter the semester.
mtpass = ["A","A-","B+","B","B-","C+","C","S"]
mtfail = ["C-","D+","D","F","NF","SF","U","W"]

In [42]:
#Section 2c - Pass/Fail Conditions & Outcome
#Purpose - To create the pass/fail conditions and outcome for the creation of a column that defines what the grade outcome was
mtstatus = [
    mtfullnodrop["Mid Term Grade"].isin(mtpass),
    mtfullnodrop["Mid Term Grade"].isin(mtfail)
]

mtout = ["MT C or Higher","MT C-, D, F, W"]

In [43]:
#Section 2d - Adding the MT Status Column
#Purpose - Using the conditions/outcome above, determines if the MT grade is passing, failing or not reported.
mtfullnodrop["MT Status"] = np.select(mtstatus, mtout, "MT Not Reported")
mtfullnodrop.head()

,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College,Mid Term Grade Number,Final Grade Number,MT Status
0,0,Registered,AED,22860,KC,A-,B,ARCH,ID,202280,FR,AED 22860,CAED,CAED,3.0,3.7,MT C or Higher
1,1,Registered,AED,22860,KC,A,A,ARCH,ID,202280,FR,AED 22860,CAED,CAED,4.0,4.0,MT C or Higher
2,2,Registered,AED,22860,KC,A,A,ARCH,OTH,202280,JR,AED 22860,Other,CAED,4.0,4.0,MT C or Higher
3,3,Registered,AED,22860,KC,F,B,ARCH,ID,202280,SO,AED 22860,CAED,CAED,3.0,0.0,MT C or Higher
4,4,Registered,AED,22860,KC,B,B,ARCH,ID,202280,FR,AED 22860,CAED,CAED,3.0,3.0,MT C or Higher


In [44]:
#Section 2e - Creating Values by Course, Term & Status
#Purpose - To create observations for each course, term, subject college, and status. This allows for the values to be pulled later
mtsummary = mtfullnodrop.value_counts(["Course Code","Academic Period","Subject","Subject College","Department","MT Status"]).reset_index()
mtsummary.head()

,Course Code,Academic Period,Subject,Subject College,Department,MT Status,count
0,COMM 17591,202580,COMM,CCI,COMM,MT C or Higher,1118
1,COMM 17591,202480,COMM,CCI,COMM,MT C or Higher,1080
2,COMM 17591,202280,COMM,CCI,COMM,MT C or Higher,994
3,COMM 17591,202380,COMM,CCI,COMM,MT C or Higher,981
4,COMM 17591,202610,COMM,CCI,COMM,MT C or Higher,532


In [45]:
#Section 2f - Creating the Summary Table
#Purpose - To further refine the summary data by pivoting the table. This allows for N/A values to be accounted for and create an observation for each course/term combo that captures all statuses
mtsummary = mtsummary.pivot(index=["Course Code", "Academic Period","Subject","Subject College","Department"],
                            columns = "MT Status",
                            values = "count",).fillna(0).reset_index()

mtsummary.head()

MT Status,Course Code,Academic Period,Subject,Subject College,Department,MT C or Higher,"MT C-, D, F, W",MT Not Reported
0,AED 22860,202280,AED,CAED,ARCH,84.0,6.0,13.0
1,AED 22860,202310,AED,CAED,ARCH,8.0,5.0,1.0
2,AED 22860,202380,AED,CAED,ARCH,58.0,12.0,9.0
3,AED 22860,202410,AED,CAED,ARCH,10.0,1.0,1.0
4,AED 22860,202480,AED,CAED,ARCH,75.0,11.0,7.0


In [46]:
#Section 2g - Creating a Total Grades Column
#Purpose - To add a total grades column. This will be used to show the total number of MT grades, but also to determine how many passing grades there are.
mtsummary["Total Grades"] = mtsummary["MT C or Higher"] + mtsummary["MT C-, D, F, W"] + mtsummary["MT Not Reported"]
mtsummary.head()

MT Status,Course Code,Academic Period,Subject,Subject College,Department,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades
0,AED 22860,202280,AED,CAED,ARCH,84.0,6.0,13.0,103.0
1,AED 22860,202310,AED,CAED,ARCH,8.0,5.0,1.0,14.0
2,AED 22860,202380,AED,CAED,ARCH,58.0,12.0,9.0,79.0
3,AED 22860,202410,AED,CAED,ARCH,10.0,1.0,1.0,12.0
4,AED 22860,202480,AED,CAED,ARCH,75.0,11.0,7.0,93.0


In [47]:
#Section 2h - Creating a # of Passing Grades Column
#Purpose - To create a sort the % of passing grades. This version of the column is a #, not a % - this is important for future calculations
mtsummary["# of Passing Grades"] = mtsummary["MT C or Higher"]/mtsummary["Total Grades"]
mtsummary = mtsummary.sort_values("# of Passing Grades", ascending = False)

In [48]:
#Section 2i - Creating a % of Passing Grades Column
#Purpose - to convert the prior column into a % - this is what will be shown on the table
mtsummary["% of Passing Grades"] = mtsummary["# of Passing Grades"].apply("{:.2%}".format)
mtsummary.head()

MT Status,Course Code,Academic Period,Subject,Subject College,Department,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades,# of Passing Grades,% of Passing Grades
689,THEA 28661,202410,THEA,CotA,THDN,5.0,0.0,0.0,5.0,1.0,100.00%
431,MDJ 11788,202380,MDJ,CCI,MDJ,19.0,0.0,0.0,19.0,1.0,100.00%
219,DAN 22742,202480,DAN,CotA,THDN,9.0,0.0,0.0,9.0,1.0,100.00%
617,THEA 14873,202480,THEA,CotA,THDN,11.0,0.0,0.0,11.0,1.0,100.00%
571,MUS 20682,202480,MUS,CotA,MUS,9.0,0.0,0.0,9.0,1.0,100.00%


##### *Creating Summary Table Variables*

In [49]:
#Section 2j - Creating Term Variable
#Purpose - To define the current term and create up to date tables - additional coding helps create future language to be used in titles
currentterm = [202610] #IMPORTANT - UPDATE FOR EACH NEW CYCLE
termmap = {10: "Spring", 60: "Summer", 80: "Fall"} #Reads the last two digits to determine term used.
year = str(currentterm[0])[:4] #reads the first 4 digits to determine the term
termsuffix = int(str(currentterm[0])[4:]) #reads the last two digits to determine the semester
prettyterm = f"{termmap.get(termsuffix, 'Semester')} {year}" #This uses the above to spit out the term that is being examined in a readable format

In [50]:
#Section 2k - Creating College Variables
#Purpose - To create the variables that will be used to make the summary tables for use in the college-specific tables
caedcheck = (mtsummary["Subject College"] == "CAED") & (mtsummary["Academic Period"].isin(currentterm))
ccicheck= (mtsummary["Subject College"] == "CCI") & (mtsummary["Academic Period"].isin(currentterm))
cotacheck= (mtsummary["Subject College"] == "CotA") & (mtsummary["Academic Period"].isin(currentterm))

In [51]:
#Section 2l - Making College-Specific Summary Tables & Values
#Purpose - To call these tables and unique values when creating the different tables later on in the code.
caedsummary = mtsummary[caedcheck]
caedsubject = sorted(caedsummary["Subject"].unique())
ccisummary = mtsummary[ccicheck]
ccischool = sorted(ccisummary["Department"].unique())
cotasummary = mtsummary[cotacheck]
cotaschool = sorted(cotasummary["Department"].unique())

In [52]:
#Section 2m - Making Column Header & Value Variables
#Purpose - To make writing the code easier, defining variables for the headers and values (all tables use the same)
headers = ["Course Code", "Total", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"]
values = ["Course Code", "Total Grades", "MT C-, D, F, W", "MT Not Reported", "MT C or Higher", "% of Passing Grades"]

In [53]:
#Section 2m - Making College-Specific Pass Rate Colors
#Purpose - To call specific values from each table to color code the pass rates (used for the main tables only).
caedpass = []
for num in caedsummary["# of Passing Grades"]:
    if num >= .85:
        caedpass.append("#CCFFCC")
    elif num >= .80:
        caedpass.append("#FFFFCC")
    else:
        caedpass.append("#FFCCCC")

ccipass = []
for num in ccisummary["# of Passing Grades"]:
    if num >= .85:
        ccipass.append("#CCFFCC")
    elif num >= .80:
        ccipass.append("#FFFFCC")
    else:
        ccipass.append("#FFCCCC")

cotapass = []
for num in cotasummary["# of Passing Grades"]:
    if num >= .85:
        cotapass.append("#CCFFCC")
    elif num >= .80:
        cotapass.append("#FFFFCC")
    else:
        cotapass.append("#FFCCCC")

In [54]:
#Section 2n - Making Zebra Rows for Each College
#Purpose - To create zebra rows for each college - important to define separately to maintain color scheme for each unit
caedrows = len(caedsummary)
caedzebra = ["#FFFFFF", "#E3E6E8"] * (caedrows // 2+1) #Use #3B4145 for the header
caedzebra = caedzebra[:caedrows]

ccirows = len(ccisummary)
ccizebra = ["#FFFFFF", "#CCE5FF"] * (ccirows // 2+1) #Use #003976 for the header
ccizebra = ccizebra[:ccirows]

cotarows = len(cotasummary)
cotazebra = ["#FFFFFF", "#FFF1CC"] * (cotarows // 2+1) #Use #EFAB00 for the header
cotazebra = cotazebra[:cotarows]

##### *Creating Summary Tables*

In [55]:
#Section 2o - CAED Table
#Purpose - To create a table that examines the MT passing rates for CAED each semester
caedtraces = []
caedbuttons = []

#This is the stuff for the "all state"
caedtraces.append(go.Table(
    header = dict(values = headers,
                  fill_color = "#3B4145",
                  line_color = "black",
                  font_color = "white",
                  font_weight = "bold",
                  font_size = 12),
    cells = dict(values = [caedsummary[v] for v in values],
                 fill_color = [caedzebra, caedzebra, caedzebra, caedzebra, caedzebra, caedpass],
                 line_color = "black",
                 font_size = 12),
    visible = True                  
))

caedbuttons.append(dict(
    label = "All",
    method = "update",
    args = [{"visible":[True] + [False] * len(caedsubject)}]#Helps make all the buttons visible and defaults the first trace to be visible
))

#This is the stuff for the different subjects
for i, caedsub in enumerate(caedsubject): #enumerate runs through the list (basically calls out each unique value in order)
    caedsubjects = caedsummary[caedsummary["Subject"] == caedsub].copy()
    caedsubjects = caedsubjects.sort_values(by = "% of Passing Grades", ascending = False) #Resorts this column so it is in numerical order
    caedsubrows = len(caedsubjects)
    caedsubpass = []

    for num in caedsubjects["# of Passing Grades"]: #I need to call this again in the loop to make the passing colors work each cycle
        if num >= .85:
            caedsubpass.append("#CCFFCC")
        elif num >= .80:
            caedsubpass.append("#FFFFCC")
        else:
            caedsubpass.append("#FFCCCC")

    caedtraces.append(go.Table(
        header = dict(values = headers,
                  line_color = "lightslategray",
                  fill_color = "#3B4145",
                  font_color = "white",
                  font_weight = "bold",
                  font_size = 12),
        cells = dict(values = [caedsubjects[v] for v in values],
                 fill_color = [caedzebra, caedzebra, caedzebra, caedzebra, caedzebra, caedsubpass],
                 line_color = "black"),
        visible = False          
    ))

    caedsubvisibility = [False] * (len(caedsubject) + 1) #Since we are making a single table, we are calling this here to make all of them appear as false
    caedsubvisibility[i + 1] = True #This makes the one you select visible

    caedbuttons.append(dict(
        label = caedsub,
        method = "update",
        args = [{"visible":caedsubvisibility}]
    ))

caedsumlayout = go.Layout(
    title=dict(text= f"<b>{prettyterm} CAED Midterm Report",
                font_weight = "bold", 
                font_color = "black", 
                xanchor = "center", 
                yanchor = "top", 
                x = .5,
                y = .965),
    height = 650,
    width = 1200
)

caedsumfig = go.Figure(data = caedtraces, layout = caedsumlayout)

caedsumfig.update_layout(updatemenus = [dict(
    type = "dropdown",
    direction = "down",
    showactive = True,
    xanchor = "center",
    yanchor = "top",
    x = .5,
    y = 1.1,
    buttons = caedbuttons)],
    annotations=[dict(
        text = "Subject:", 
        showarrow = False,
        xanchor = "center",
        yanchor = "top",
        x = .435,
        y = 1.08
    )])

caedsumfig.show()

In [56]:
#Section 2p - CCI Table
#Purpose - To create a table that examines the MT passing rates for CCI each semester
ccitraces = []
ccibuttons = []

ccitraces.append(go.Table(
    header = dict(values = headers,
                  line_color = "black",
                  fill_color = "#003976",
                  font_color = "white",
                  font_weight = "bold",
                  font_size = 12),
    cells = dict(values = [ccisummary[v] for v in values],
                 fill_color = [ccizebra, ccizebra, ccizebra, ccizebra, ccizebra, ccipass],
                 line_color = "black",
                 font_size = 12),
    visible = True                  
))

ccibuttons.append(dict(
    label = "All",
    method = "update",
    args = [{"visible":[True] + [False] * len(ccischool)}]
))

for i, ccisch in enumerate(ccischool):
    ccischools = ccisummary[ccisummary["Subject"] == ccisch].copy()
    ccischools = ccischools.sort_values(by = "% of Passing Grades", ascending = False)
    ccischrows = len(ccischools)
    ccischpass = []

    for num in ccischools["# of Passing Grades"]:
        if num >= .85:    
            ccischpass.append("#CCFFCC")
        elif num >= .80:
            ccischpass.append("#FFFFCC")
        else:
            ccischpass.append("#FFCCCC")

    ccitraces.append(go.Table(
        header = dict(values = headers,
                  line_color = "black",
                  fill_color = "#003976",
                  font_color = "white",
                  font_weight = "bold",
                  font_size = 12),
        cells = dict(values = [ccischools[v] for v in values],
                 fill_color = [ccizebra, ccizebra, ccizebra, ccizebra, ccizebra, ccischpass],
                 line_color = "black"),
        visible = False          
    ))

    ccischvisibility = [False] * (len(ccischool) + 1)
    ccischvisibility[i + 1] = True

    ccibuttons.append(dict(
        label = ccisch,
        method = "update",
        args = [{"visible":ccischvisibility}]
    ))

ccisumlayout = go.Layout(
    title=dict(text= f"<b>{prettyterm} CCI Midterm Report",
                font_weight = "bold", 
                font_color = "black", 
                xanchor = "center", 
                yanchor = "top", 
                x = .5,
                y = .965),
    height = 650,
    width = 1200
)

ccisumfig = go.Figure(data = ccitraces, layout = ccisumlayout)

ccisumfig.update_layout(updatemenus = [dict(
    type = "dropdown",
    direction = "down",
    showactive = True,
    xanchor = "center",
    yanchor = "top",
    x = .5,
    y = 1.1,
    buttons = ccibuttons)],
    annotations=[dict(
        text = "School:", 
        showarrow = False,
        xanchor = "center",
        yanchor = "top",
        x = .435,
        y = 1.08
    )])

ccisumfig.show()

In [57]:
#Section 2p - CotA Table
#Purpose - To create a table that examines the MT passing rates for CotA each semester
cotatraces = []
cotabuttons = []

cotatraces.append(go.Table(
    header = dict(values = headers,
                  line_color = "black",
                  fill_color = "#EFAB00",
                  font_color = "black",
                  font_weight = "bold",
                  font_size = 12),
    cells = dict(values = [cotasummary[v] for v in values],
                 fill_color = [cotazebra, cotazebra, cotazebra, cotazebra, cotazebra, cotapass],
                 line_color = "black",
                 font_size = 12),
    visible = True                  
))

cotabuttons.append(dict(
    label = "All",
    method = "update",
    args = [{"visible":[True] + [False] * len(cotaschool)}]
))

for i, cotasch in enumerate(cotaschool):
    cotaschools = cotasummary[cotasummary["Department"] == cotasch].copy()
    cotaschools = cotaschools.sort_values(by = "% of Passing Grades", ascending = False)
    cotaschrows = len(cotaschools)
    cotaschpass = []

    for num in cotaschools["# of Passing Grades"]:
        if num >= .85:    
            cotaschpass.append("#CCFFCC")
        elif num >= .80:
            cotaschpass.append("#FFFFCC")
        else:
            cotaschpass.append("#FFCCCC")

    cotatraces.append(go.Table(
        header = dict(values = headers,
                  line_color = "black",
                  fill_color = "#EFAB00",
                  font_color = "black",
                  font_weight = "bold",
                  font_size = 12),
        cells = dict(values = [cotaschools[v] for v in values],
                 fill_color = [cotazebra, cotazebra, cotazebra, cotazebra, cotazebra, cotaschpass],
                 line_color = "black"),
        visible = False          
    ))

    cotaschvisibility = [False] * (len(cotaschool) + 1)
    cotaschvisibility[i + 1] = True

    cotabuttons.append(dict(
        label = cotasch,
        method = "update",
        args = [{"visible":cotaschvisibility}]
    ))

cotasumlayout = go.Layout(
    title=dict(text= f"<b>{prettyterm} CotA Midterm Report",
                font_weight = "bold", 
                font_color = "black", 
                xanchor = "center", 
                yanchor = "top", 
                x = .5,
                y = .965),
    height = 650,
    width = 1200
)

cotasumfig = go.Figure(data = cotatraces, layout = cotasumlayout)

cotasumfig.update_layout(updatemenus = [dict(
    type = "dropdown",
    direction = "down",
    showactive = True,
    xanchor = "center",
    yanchor = "top",
    x = .5,
    y = 1.1,
    buttons = cotabuttons)],
    annotations=[dict(
        text = "School:", 
        showarrow = False,
        xanchor = "center",
        yanchor = "top",
        x = .435,
        y = 1.08
    )])

cotasumfig.show()

#### **Section 3 - Intervention Summary Table**

This section covers the steps to make the a table that summarizes the needed number of interventions by the academic advisors.

In [58]:
#Section 3a - Create Intervention Variable
#Purpose - To create the intervention variable
mtinterfull = pd.read_csv("mthubinterventions.csv")
mtinterfull.head()

,Unnamed: 0,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College,Mid Term Grade Number,Final Grade Number,MT Intervention Needed
0,11,11,Registered,AED,22860,KC,NaN,C,ARCH,ID,202280,FR,AED 22860,CAED,CAED,2.0,NaN,FR/SO Intervention Needed
1,14,14,Registered,AED,22860,KC,C,D,ARCH,ID,202280,FR,AED 22860,CAED,CAED,1.0,2.0,FR/SO Intervention Needed
2,19,19,Registered,AED,22860,KC,B-,C,ARCH,ID,202280,SO,AED 22860,CAED,CAED,2.0,2.7,FR/SO Intervention Needed
3,31,31,Stopped Attending - Failed,AED,22860,KC,W,D,ARCH,ID,202280,SO,AED 22860,CAED,CAED,1.0,0.0,FR/SO Intervention Needed
4,33,33,Registered,AED,22860,KC,B-,C-,ARCH,ID,202280,FR,AED 22860,CAED,CAED,1.7,2.7,FR/SO Intervention Needed


In [59]:
#Section 3b - Generating Counts
#Purpose - To generate count of interventions to be used in future function
mtinterfull["MT Intervention Needed"].value_counts()

MT Intervention Needed
FR/SO Intervention Needed    5460
JR/SR Intervention Needed     652
Name: count, dtype: int64

In [60]:
#Section 3c - Creating the Pivot Table
#Purpose - 
intercountpivot = mtinterfull.pivot_table(index=["Academic Period", "Major College"],
                            columns = "MT Intervention Needed",
                            values = "Record ID",
                            aggfunc = "count").reset_index()

intercountpivot.head()

MT Intervention Needed,Academic Period,Major College,FR/SO Intervention Needed,JR/SR Intervention Needed
0,202280,CAED,133,15
1,202280,CCI,209,49
2,202280,CotA,424,37
3,202310,CAED,122,18
4,202310,CCI,155,32


In [61]:
#Section 3d - Adding Total Instances Column
#Purpose - 
intercountpivot["Total # of Instances"] = intercountpivot["FR/SO Intervention Needed"] + intercountpivot["JR/SR Intervention Needed"]
intercountpivot.head()

MT Intervention Needed,Academic Period,Major College,FR/SO Intervention Needed,JR/SR Intervention Needed,Total # of Instances
0,202280,CAED,133,15,148
1,202280,CCI,209,49,258
2,202280,CotA,424,37,461
3,202310,CAED,122,18,140
4,202310,CCI,155,32,187


##### *Making Intervention Table Variables*

In [62]:
#Section 3e - Creating College Logic
#Purpose - 
caedintercheck = (intercountpivot["Major College"] == "CAED") & (intercountpivot["Academic Period"].isin(currentterm))
cciintercheck = (intercountpivot["Major College"] == "CCI") & (intercountpivot["Academic Period"].isin(currentterm))
cotaintercheck = (intercountpivot["Major College"] == "CotA") & (intercountpivot["Academic Period"].isin(currentterm))

In [63]:
#Section 3f - Making Intervention Dataframes
#Purpose - 
caedinter = intercountpivot[caedintercheck]
cciinter = intercountpivot[cciintercheck]
cotainter = intercountpivot[cotaintercheck]

##### *Creating Advisor Intervention Rate Tables*

In [64]:
#Section 3h - CAED Table
#Purpose - 

caedintertrace = go.Table(
    header=dict(values = ["Group #", "CAED Midterm Intervention - Student Criteria", "# of Instances"],                
                line_color = "black",
                fill_color = "#3B4145",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values = [
                [1,2," "],
                 ["CAED Major, FR or SO, CAED Course, midterm D+ or Below, still registered", 
                  "CAED Major, JR or SR, CAED Course, midterm F, still registered",
                  "Total"],
                 [caedinter["FR/SO Intervention Needed"], caedinter["JR/SR Intervention Needed"], caedinter["Total # of Instances"]]],
                 line_color = "black",
                 fill_color = [["#FFFFFF", "#E3E6E8", "#3B4145"], ["#FFFFFF", "#E3E6E8", "#3B4145"], ["#FFFFFF", "#E3E6E8", "#3B4145"]],
                 font_color = [["black", "black", "#FFFFFF"], ["black", "black", "#FFFFFF"], ["black", "black", "#FFFFFF"]]),
    visible = True)

interlayout = go.Layout(
    title=dict(text=f"<b>{prettyterm} CAED Advisor Intervention Table",
            font_weight = "bold", 
            xanchor = "center",
            x = .5),
    height = 400,
    width = 1200
)

caedinterfig = go.Figure(data = caedintertrace, layout=interlayout)

caedinterfig.show()

In [65]:
#Section 3i - CCI Table
#Purpose -

cciintertrace = go.Table(
    header=dict(values = ["Group #", "CCI Midterm Intervention - Student Criteria", "# of Instances"],                
                line_color = "black",
                fill_color = "#003976",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values = [
                [1,2," "],
                 ["CCI Major, FR or SO, CAED Course, midterm D+ or Below, still registered", 
                  "CCI Major, JR or SR, CAED Course, midterm F, still registered",
                  "Total"],
                 [cciinter["FR/SO Intervention Needed"], cciinter["JR/SR Intervention Needed"], cciinter["Total # of Instances"]]],
                 line_color = "black",
                 fill_color = [["#FFFFFF", "#CCE5FF", "#003976"], ["#FFFFFF", "#CCE5FF", "#003976"], ["#FFFFFF", "#CCE5FF", "#003976"]],
                 font_color = [["black", "black", "#FFFFFF"], ["black", "black", "#FFFFFF"], ["black", "black", "#FFFFFF"]]),
    visible = True)

interlayout = go.Layout(
    title=dict(text=f"<b>{prettyterm} CCI Advisor Intervention Table",
            font_weight = "bold", 
            xanchor = "center",
            x = .5),
    height = 400,
    width = 1200
)

cciinterfig = go.Figure(data = cciintertrace, layout=interlayout)

cciinterfig.show()

In [66]:
#Section 3i - CotA Table
#Purpose -

cotaintertrace = go.Table(
    header=dict(values = ["Group #", "CotA Midterm Intervention - Student Criteria", "# of Instances"],                
                line_color = "black",
                fill_color = "#EFAB00",
                font_color = "white",
                font_weight = "bold"),
    cells = dict(values = [
                [1,2," "],
                 ["CotA Major, FR or SO, CotA Course, midterm D+ or Below, still registered", 
                  "CotA Major, JR or SR, CotA Course, midterm F, still registered",
                  "Total"],
                 [cotainter["FR/SO Intervention Needed"], cotainter["JR/SR Intervention Needed"], cotainter["Total # of Instances"]]],
                 line_color = "black",
                 fill_color = [["#FFFFFF", "#FFF1CC", "#EFAB00"], ["#FFF1CC", "#FFF1CC", "#EFAB00"], ["#FFFFFF", "#FFF1CC", "#EFAB00"]],
                 font_color = "black"),
    visible = True)

interlayout = go.Layout(
    title=dict(text=f"<b>{prettyterm} CotA Advisor Intervention Table",
                font_weight = "bold", 
                            xanchor = "center",
                                        x = .5),
    height = 400,
    width = 1200
)

cotainterfig = go.Figure(data = cotaintertrace, layout=interlayout)

cotainterfig.show()

#### **Section 4 - Intervention Contact Sheet**

This section is dedicated to the creation of the intervention contact sheet for academic advisors. This sheet will allow advisors know who to contact, their contact information and what course(s) indicate a failing MT grade. As the advisors operate under a hub model, a single Excel file will be created with tabs for each college. The advising team may download this sheet and create a shared copy that they can reference to track which student(s) have been contacted.

##### *Creating the Intervention Contact DataFrame*

In [67]:
#Section 4a - 
#Purpose - 
intercontact = pd.read_csv("mthubinterventions.csv")
intercontact.head()

,Unnamed: 0,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College,Mid Term Grade Number,Final Grade Number,MT Intervention Needed
0,11,11,Registered,AED,22860,KC,NaN,C,ARCH,ID,202280,FR,AED 22860,CAED,CAED,2.0,NaN,FR/SO Intervention Needed
1,14,14,Registered,AED,22860,KC,C,D,ARCH,ID,202280,FR,AED 22860,CAED,CAED,1.0,2.0,FR/SO Intervention Needed
2,19,19,Registered,AED,22860,KC,B-,C,ARCH,ID,202280,SO,AED 22860,CAED,CAED,2.0,2.7,FR/SO Intervention Needed
3,31,31,Stopped Attending - Failed,AED,22860,KC,W,D,ARCH,ID,202280,SO,AED 22860,CAED,CAED,1.0,0.0,FR/SO Intervention Needed
4,33,33,Registered,AED,22860,KC,B-,C-,ARCH,ID,202280,FR,AED 22860,CAED,CAED,1.7,2.7,FR/SO Intervention Needed


In [68]:
#Section 4b - 
#Purpose - 
intercontact["MT Intervention Group"] = np.where(intercontact["MT Intervention Needed"] == "FR/SO Intervention Needed", 1, 2)
intercontact.head()

,Unnamed: 0,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,Subject College,Mid Term Grade Number,Final Grade Number,MT Intervention Needed,MT Intervention Group
0,11,11,Registered,AED,22860,KC,NaN,C,ARCH,ID,202280,FR,AED 22860,CAED,CAED,2.0,NaN,FR/SO Intervention Needed,1
1,14,14,Registered,AED,22860,KC,C,D,ARCH,ID,202280,FR,AED 22860,CAED,CAED,1.0,2.0,FR/SO Intervention Needed,1
2,19,19,Registered,AED,22860,KC,B-,C,ARCH,ID,202280,SO,AED 22860,CAED,CAED,2.0,2.7,FR/SO Intervention Needed,1
3,31,31,Stopped Attending - Failed,AED,22860,KC,W,D,ARCH,ID,202280,SO,AED 22860,CAED,CAED,1.0,0.0,FR/SO Intervention Needed,1
4,33,33,Registered,AED,22860,KC,B-,C-,ARCH,ID,202280,FR,AED 22860,CAED,CAED,1.7,2.7,FR/SO Intervention Needed,1


In [69]:
#Section 4c - 
#Purpose - 
intercontact[["Notes", "Advising Pin", "Hold", "Advisor", "Date"]] = np.nan
intercontact.head()

,Unnamed: 0,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,...,Subject College,Mid Term Grade Number,Final Grade Number,MT Intervention Needed,MT Intervention Group,Notes,Advising Pin,Hold,Advisor,Date
0,11,11,Registered,AED,22860,KC,NaN,C,ARCH,ID,...,CAED,2.0,NaN,FR/SO Intervention Needed,1,NaN,NaN,NaN,NaN,NaN
1,14,14,Registered,AED,22860,KC,C,D,ARCH,ID,...,CAED,1.0,2.0,FR/SO Intervention Needed,1,NaN,NaN,NaN,NaN,NaN
2,19,19,Registered,AED,22860,KC,B-,C,ARCH,ID,...,CAED,2.0,2.7,FR/SO Intervention Needed,1,NaN,NaN,NaN,NaN,NaN
3,31,31,Stopped Attending - Failed,AED,22860,KC,W,D,ARCH,ID,...,CAED,1.0,0.0,FR/SO Intervention Needed,1,NaN,NaN,NaN,NaN,NaN
4,33,33,Registered,AED,22860,KC,B-,C-,ARCH,ID,...,CAED,1.7,2.7,FR/SO Intervention Needed,1,NaN,NaN,NaN,NaN,NaN


In [70]:
#Section 4d -
#Purpose - 
intercontactdrop = intercontact.drop(columns = ["Unnamed: 0", 'Final Grade', "Subject College", 'Final Grade Number', 'Mid Term Grade Number', 'MT Intervention Needed', 'Registration Status','Campus'], axis=0)
intercontactdrop

,Record ID,Subject,Course,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,MT Intervention Group,Notes,Advising Pin,Hold,Advisor,Date
0,11,AED,22860,C,ARCH,ID,202280,FR,AED 22860,CAED,1,NaN,NaN,NaN,NaN,NaN
1,14,AED,22860,D,ARCH,ID,202280,FR,AED 22860,CAED,1,NaN,NaN,NaN,NaN,NaN
2,19,AED,22860,C,ARCH,ID,202280,SO,AED 22860,CAED,1,NaN,NaN,NaN,NaN,NaN
3,31,AED,22860,D,ARCH,ID,202280,SO,AED 22860,CAED,1,NaN,NaN,NaN,NaN,NaN
4,33,AED,22860,C-,ARCH,ID,202280,FR,AED 22860,CAED,1,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6107,85275,MDJ,20288,F,MDJ,FM,202280,SR,MDJ 20288,CotA,2,NaN,NaN,NaN,NaN,NaN
6108,85309,MDJ,20288,F,MDJ,DMP,202310,JR,MDJ 20288,CCI,2,NaN,NaN,NaN,NaN,NaN
6109,85315,MDJ,20288,D,MDJ,FM,202310,SO,MDJ 20288,CotA,1,NaN,NaN,NaN,NaN,NaN
6110,85316,MDJ,20288,C-,MDJ,FM,202310,SO,MDJ 20288,CotA,1,NaN,NaN,NaN,NaN,NaN


In [71]:
#Section 4e - 
#Purpose - 
intercontactdrop = intercontactdrop.rename(columns = {"Record ID":"ID"})
intercontactdrop

,ID,Subject,Course,Mid Term Grade,Department,Major,Academic Period,Class,Course Code,Major College,MT Intervention Group,Notes,Advising Pin,Hold,Advisor,Date
0,11,AED,22860,C,ARCH,ID,202280,FR,AED 22860,CAED,1,NaN,NaN,NaN,NaN,NaN
1,14,AED,22860,D,ARCH,ID,202280,FR,AED 22860,CAED,1,NaN,NaN,NaN,NaN,NaN
2,19,AED,22860,C,ARCH,ID,202280,SO,AED 22860,CAED,1,NaN,NaN,NaN,NaN,NaN
3,31,AED,22860,D,ARCH,ID,202280,SO,AED 22860,CAED,1,NaN,NaN,NaN,NaN,NaN
4,33,AED,22860,C-,ARCH,ID,202280,FR,AED 22860,CAED,1,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6107,85275,MDJ,20288,F,MDJ,FM,202280,SR,MDJ 20288,CotA,2,NaN,NaN,NaN,NaN,NaN
6108,85309,MDJ,20288,F,MDJ,DMP,202310,JR,MDJ 20288,CCI,2,NaN,NaN,NaN,NaN,NaN
6109,85315,MDJ,20288,D,MDJ,FM,202310,SO,MDJ 20288,CotA,1,NaN,NaN,NaN,NaN,NaN
6110,85316,MDJ,20288,C-,MDJ,FM,202310,SO,MDJ 20288,CotA,1,NaN,NaN,NaN,NaN,NaN


In [72]:
#Section 4f - 
#Purpose - 
intercontactorder = intercontactdrop[['MT Intervention Group','Notes','Advising Pin','Hold','Advisor', 'Date','Major','Class','Mid Term Grade','Department','Subject','Course','ID','Academic Period','Major College']]
intercontactorder.head()

,MT Intervention Group,Notes,Advising Pin,Hold,Advisor,Date,Major,Class,Mid Term Grade,Department,Subject,Course,ID,Academic Period,Major College
0,1,NaN,NaN,NaN,NaN,NaN,ID,FR,C,ARCH,AED,22860,11,202280,CAED
1,1,NaN,NaN,NaN,NaN,NaN,ID,FR,D,ARCH,AED,22860,14,202280,CAED
2,1,NaN,NaN,NaN,NaN,NaN,ID,SO,C,ARCH,AED,22860,19,202280,CAED
3,1,NaN,NaN,NaN,NaN,NaN,ID,SO,D,ARCH,AED,22860,31,202280,CAED
4,1,NaN,NaN,NaN,NaN,NaN,ID,FR,C-,ARCH,AED,22860,33,202280,CAED


In [73]:
#Section 4g -
#Purpose - 
intercontactorder = intercontactorder.sort_values(by="MT Intervention Group")
intercontactorder

,MT Intervention Group,Notes,Advising Pin,Hold,Advisor,Date,Major,Class,Mid Term Grade,Department,Subject,Course,ID,Academic Period,Major College
0,1,NaN,NaN,NaN,NaN,NaN,ID,FR,C,ARCH,AED,22860,11,202280,CAED
1,1,NaN,NaN,NaN,NaN,NaN,ID,FR,D,ARCH,AED,22860,14,202280,CAED
2,1,NaN,NaN,NaN,NaN,NaN,ID,SO,C,ARCH,AED,22860,19,202280,CAED
3,1,NaN,NaN,NaN,NaN,NaN,ID,SO,D,ARCH,AED,22860,31,202280,CAED
4,1,NaN,NaN,NaN,NaN,NaN,ID,FR,C-,ARCH,AED,22860,33,202280,CAED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6107,2,NaN,NaN,NaN,NaN,NaN,FM,SR,F,MDJ,MDJ,20288,85275,202280,CotA
6108,2,NaN,NaN,NaN,NaN,NaN,DMP,JR,F,MDJ,MDJ,20288,85309,202310,CCI
3641,2,NaN,NaN,NaN,NaN,NaN,FM,SR,SF,FDM,FDM,20907,49404,202510,CotA
3693,2,NaN,NaN,NaN,NaN,NaN,FD,SR,F,FDM,FDM,27868,49928,202410,CotA


##### *Making Intervention Contact Variables*

In [74]:
#Section 4h - 
#Purpose - 
caedintercontactcheck = (intercontactorder["Major College"] == "CAED") & (intercontactorder["Academic Period"].isin(currentterm))
cciintercontactcheck = (intercontactorder["Major College"] == "CCI") & (intercontactorder["Academic Period"].isin(currentterm))
cotaintercontactcheck = (intercontactorder["Major College"] == "CotA") & (intercontactorder["Academic Period"].isin(currentterm))

In [75]:
#Section 4i -
#Purpose - 
caedcontacts = intercontactorder[caedintercontactcheck]
ccicontacts = intercontactorder[cciintercontactcheck]
cotacontacts = intercontactorder[cotaintercontactcheck]

##### *Creating the Intervention Contact Excel File*

In [76]:
#Section 4j - 
#Purpose
interwriter = pd.ExcelWriter("Spring_2026_Midterm_Intervention_List.xlsx",
                             engine = "xlsxwriter")

caedcontacts.to_excel(interwriter, index = False, sheet_name = "CAEDInterventions")
ccicontacts.to_excel(interwriter, index = False, sheet_name = "CCIInterventions")
cotacontacts.to_excel(interwriter, index = False, sheet_name = "CotAInterventions")

In [77]:
#Section 4k -
#Purpose - 
interbook = interwriter.book
caedintersheet = interwriter.sheets["CAEDInterventions"]
cciintersheet = interwriter.sheets["CCIInterventions"]
cotaintersheet = interwriter.sheets["CotAInterventions"]

In [78]:
#Section 4l -
#Purpose - 
header_format = interbook.add_format({
    "font_name": "Andale WT",
    "font_size": 8,
    "bold": True,
    "font_color":"#FFFFFF",
    "bg_color": "#003976",
    "align":"center"
})

#Applying the formatting to each sheet
caedcolumns = [{'header': column, "header_format":header_format} for column in caedcontacts.columns]
ccicolumns = [{'header': column, "header_format":header_format} for column in ccicontacts.columns]
cotacolumns = [{'header': column, "header_format":header_format} for column in cotacontacts.columns]

alertformat = interbook.add_format({"bg_color":'#FFCCCC',
                                    "font_name": "Andale WT",
                                    "font_size": 8,
                                    "align": "center"
})

rowcustoms = interbook.add_format({
    "font_name": "Andale WT",
    "font_size": 8,
    "align": "center"
})

In [79]:
#Section 4m - 
(caedrowmax, caedcolmax) = caedcontacts.shape

caedintersheet.add_table(0, 0, caedrowmax, caedcolmax - 1, {
    "columns": caedcolumns,
    "style": "Table Style Medium 2" 
})

caedintersheet.conditional_format(1, 6, caedrowmax, 8, {
    "type": "no_errors",
    "format": alertformat
})

0

In [80]:
#Section 4n - 
#Purpose - 
(ccirowmax, ccicolmax) = ccicontacts.shape #Setting the max/min columns

#all rows formatting
cciintersheet.add_table(0, 0, ccirowmax, ccicolmax - 1, { #Establishing the look for the whole thing
    "columns": ccicolumns, #specifying the column format
    "style": "Table Style Medium 2" # This gives light blue zebra stripes
})

# Formatting for the columns I want to highlight
cciintersheet.conditional_format(1, 6, ccirowmax, 8, {#This limits the impact to just those rows
    "type": "no_errors", #this is saying to look at all the cells in that range - just ignore the broken ones
    "format": alertformat #pull the alert format from earlier
})

0

In [81]:
#Section 4o - 
#Purpose - 
(cotarowmax, cotacolmax) = cotacontacts.shape

cotaintersheet.add_table(0, 0, cotarowmax, cotacolmax - 1, {
    "columns": cotacolumns,
    "style": "Table Style Medium 2" 
})

cotaintersheet.conditional_format(1, 6, cotarowmax, 8, {
    "type": "no_errors",
    "format": alertformat
})

0

In [82]:
#Section 4p -
#Purpose - 
caedintersheet.set_column(0, caedcolmax-1, 20, rowcustoms)
caedintersheet.set_column(1, 1, 80, rowcustoms)
caedintersheet.freeze_panes(1,0)

cciintersheet.set_column(0, ccicolmax-1, 20, rowcustoms)
cciintersheet.set_column(1, 1, 80, rowcustoms)
cciintersheet.freeze_panes(1,0)

cotaintersheet.set_column(0, cotacolmax-1, 20, rowcustoms)
cotaintersheet.set_column(1, 1, 80, rowcustoms)
cotaintersheet.freeze_panes(1,0)

interwriter.close()

#### **Section 5 - Loading Figures Into PowerBI**

##### *PowerBI Instructions*

Welcome to the most painful part of this process. This will be primarily narrative, with some isolated code (discussing what did v. did not work with it). The *successful* code can be found at the end, starting with the section labeled *CAED Figures*.

My figures will be hosted on a PowerBI dashboard. The logic behind this is that most of my pre-existing work is done in PowerBI, so my audiences are used to this format. Additionally, the university is moving to PowerBI for its reporting structure, so it is good practice for that as well. To learn about integrating plotly and Python into PowerBI, I did some reading online and consultatory work with Gemini to get it to work.

I wanted to go over what I found in my research and conversations with Gemini about how PowerBI works with plotly code. Here are a few key points:
1. PowerBI has a native Python language tool, but there are multiple visuals you can get from the Microsoft Store that work with code (HTML readers and one that lets you use a pseudo-plotly in PowerBI are some of the highlights)
2. PowerBI defaults to reading data as "datasets." You can plug dataset into your code and run it in VS Code, but it won't work. However, when you move it to PowerBI, it picks it up and *does* work. What I found recommended was calling your "base" dataset as dataset - this way your visualization can still run and preview it prior to uploading it to PowerBI.
3. You must first plug your CSV file into PowerBI in order to run the code - this is what PowerBI tries to read. If you use the native tool, you can plug values in via that and click "run" instead of having to connect to the full dataset. Alternatively, you can turn the visualization into an HTML file, and it will run that instead.

No matter what I tried, I could not get the native Python tool in PowerBI to work. It kept fighting me and making it difficult to run properly. An alternative that came up was HTML Content Viewer. HTML Content Viewer is a tool in PowerBI that can display Python visuals as HTML *if* you plug in HTML code. I though this sounded great, but my experience with HTML is minimal and (being transparent) didn't want to devote too much time to HTML right now. To this end, I did some work with Gemini trying to convert my code into HTML that would work in PowerBI. Below you will see the prompt, why it didn't work, and the code. Please note that these often go at the end of the figure, so in an effort to conserve space only the "final" bit is shown.

##### Attempt 1 - Converting Raw

**Prompt** - Can you convert my figure into an HTML file that could be uploaded?

**What Happened?** - When loaded into PowerBI, it worked - but you needed to move it first. I kept tweaking with the size (in the original code) to see if that was an issue, but that wasn't the problem. 

In [83]:
html_str = pio.to_html(caedsumfig, full_html=False, include_plotlyjs='cdn') #converting the figure into an HTML file - this is including the plotly javascript

print(html_str)

<div>                        <script>window.PlotlyConfig = {MathJaxConfig: 'local'};</script>
        <script charset="utf-8" src="https://cdn.plot.ly/plotly-3.4.0.min.js" integrity="sha256-KEmPoupLpFyGMyGAiOsiNDbKDKAvxXAn/W+oQa0ZAfk=" crossorigin="anonymous"></script>                <div id="3cb8dbb0-3c7f-4f50-ad1e-5c80091bbae7" class="plotly-graph-div" style="height:650px; width:1200px;"></div>            <script>                window.PLOTLYENV=window.PLOTLYENV || {};                                if (document.getElementById("3cb8dbb0-3c7f-4f50-ad1e-5c80091bbae7")) {                    Plotly.newPlot(                        "3cb8dbb0-3c7f-4f50-ad1e-5c80091bbae7",                        [{"cells":{"fill":{"color":[["#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF"],["#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8",

##### Attempt 2 - Resizing Automatically

**Prompt** - So when it is in there, I need to resize it to be visible. Could we build that in?

**What Happened?** - It didn't resize it - it needed a trigger time, so it never ran

In [84]:
html_str = pio.to_html(caedsumfig, full_html=False, include_plotlyjs='cdn') #converting the figure into an HTML file - this is including the plotly javascript

html_str += '<script>window.dispatchEvent(new Event("resize"));</script>'

print(html_str)

<div>                        <script>window.PlotlyConfig = {MathJaxConfig: 'local'};</script>
        <script charset="utf-8" src="https://cdn.plot.ly/plotly-3.4.0.min.js" integrity="sha256-KEmPoupLpFyGMyGAiOsiNDbKDKAvxXAn/W+oQa0ZAfk=" crossorigin="anonymous"></script>                <div id="8ca39912-0e0a-41e7-8c61-c414c331a027" class="plotly-graph-div" style="height:650px; width:1200px;"></div>            <script>                window.PLOTLYENV=window.PLOTLYENV || {};                                if (document.getElementById("8ca39912-0e0a-41e7-8c61-c414c331a027")) {                    Plotly.newPlot(                        "8ca39912-0e0a-41e7-8c61-c414c331a027",                        [{"cells":{"fill":{"color":[["#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF"],["#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8",

##### Attempt 3 - Coding the Resize to Happen on Load

**Prompt** - Can we add some form of timer or function that resizes it on load?

**What Happened?** - Didn't work - the resize didn't trigger (even on initial load)

In [85]:
html_str = pio.to_html(caedsumfig, full_html=False, include_plotlyjs='cdn')

# Add this line right here:
html_str = html_str.replace('style="height:650px; width:100%;"', 'style="height:650px; width:100%;" onload="window.dispatchEvent(new Event(\'resize\'));"')

print(html_str)

<div>                        <script>window.PlotlyConfig = {MathJaxConfig: 'local'};</script>
        <script charset="utf-8" src="https://cdn.plot.ly/plotly-3.4.0.min.js" integrity="sha256-KEmPoupLpFyGMyGAiOsiNDbKDKAvxXAn/W+oQa0ZAfk=" crossorigin="anonymous"></script>                <div id="c612e55b-6daf-4d80-bad3-ebab0e198637" class="plotly-graph-div" style="height:650px; width:1200px;"></div>            <script>                window.PLOTLYENV=window.PLOTLYENV || {};                                if (document.getElementById("c612e55b-6daf-4d80-bad3-ebab0e198637")) {                    Plotly.newPlot(                        "c612e55b-6daf-4d80-bad3-ebab0e198637",                        [{"cells":{"fill":{"color":[["#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF"],["#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8",

##### Attempt 4 - Putting a Timer In

**Prompt** - Let's add a timer - maybe if it reloads every so often it'd work?

**What Happened?** - This is actually a two in one - I kept adjusting the timer to make it work. However, when it was too low, it'd constantly reload and not let you filter since it kept refreshing. Conversely, if you set it too long, it'd never load the initial instance while you wait.

In [86]:
html_str = pio.to_html(
    caedsumfig,
    full_html=False,
    include_plotlyjs='cdn',
    config={"responsive": True}
)

html_str = f"""
<div id="plotly-div">
{html_str}
</div>

<script>
setTimeout(function() {{
    window.dispatchEvent(new Event('resize'));
}}, 300);
</script>
"""

##### Attempt 5 - The Working Attempt

**Prompt** - So the timer system doesn't work. I did some research and some people recommend converting figures to JSON files - could we try that?

**What Happened** - It actually worked. Based on what Gemini said, it works because it is rendering it directly as a pre-rendered object instead of running it in HTML. The filters and everything still work, so this is what will be copied over to the other figures we made!

The code can be seen below (in the CAED/CCI/CotA Figures sections).

*A quick note on using PowerBI* - Something I took a second to try was putting these figures into PowerBI and then uploading it to My Workspace in PowerBI. Unfortunately, HTML Content Viewer isn't a certified Microsoft program, so the university's system blocks showing it online. However, it shows it in offline versions of PowerBI. This means that I'll need to host it as a separate PowerBI file that can be accessed, possibly through my OneDrive, that users can go in and review. I'll do some more work on this as we move forward into the final project!

##### *CAED Figures*

In [87]:
#CAED MT Summary Report
caedsumjson = caedsumfig.to_json()

caedsumhtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {caedsumjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(caedsumhtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"cells":{"fill":{"color":[["#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF"],["#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF"],["#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF"],["#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF","#E3E6E8","#FFFFFF"],[

In [88]:
#CAED MT Intervention Table
caedinterjson = caedinterfig.to_json()

caedinterhtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {caedinterjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(caedinterhtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"cells":{"fill":{"color":[["#FFFFFF","#E3E6E8","#3B4145"],["#FFFFFF","#E3E6E8","#3B4145"],["#FFFFFF","#E3E6E8","#3B4145"]]},"font":{"color":[["black","black","#FFFFFF"],["black","black","#FFFFFF"],["black","black","#FFFFFF"]]},"line":{"color":"black"},"values":[[1,2," "],["CAED Major, FR or SO, CAED Course, midterm D+ or Below, still registered","CAED Major, JR or SR, CAED Course, midterm F, still registered","Total"],[[84],[14],[98]]]},"header":{"fill":{"color":"#3B4145"},"font":{"color":"white","weight":"bold"},"line":{"color":"black"},"values":["Group #","CAED Midterm Intervention - Student Criteria","# of Instances"]},"visible":true,"type":"table"}],"layout":{"height":400,"title":{"font":{"weight":"bold"},"text":"\u003cb\u003eSpring 2026 CAED Advisor Intervention Table","x":0.5,"xanchor":"center"},"width":1200,"template":{"data":{"histogram2dcontour":[{"type"

##### *CCI Figures*

In [89]:
##CCI MT Grade Summary Table
ccisumjson = ccisumfig.to_json()

ccisumhtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {ccisumjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(ccisumhtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"cells":{"fill":{"color":[["#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF"],["#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF"],["#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF"],["#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#FFFFFF","#CCE5FF","#

In [90]:
#CCI Interventions Summary Table
cciinterjson = cciinterfig.to_json()

cciinterhtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {cciinterjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(cciinterhtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"cells":{"fill":{"color":[["#FFFFFF","#CCE5FF","#003976"],["#FFFFFF","#CCE5FF","#003976"],["#FFFFFF","#CCE5FF","#003976"]]},"font":{"color":[["black","black","#FFFFFF"],["black","black","#FFFFFF"],["black","black","#FFFFFF"]]},"line":{"color":"black"},"values":[[1,2," "],["CCI Major, FR or SO, CAED Course, midterm D+ or Below, still registered","CCI Major, JR or SR, CAED Course, midterm F, still registered","Total"],[[72],[19],[91]]]},"header":{"fill":{"color":"#003976"},"font":{"color":"white","weight":"bold"},"line":{"color":"black"},"values":["Group #","CCI Midterm Intervention - Student Criteria","# of Instances"]},"visible":true,"type":"table"}],"layout":{"height":400,"title":{"font":{"weight":"bold"},"text":"\u003cb\u003eSpring 2026 CCI Advisor Intervention Table","x":0.5,"xanchor":"center"},"width":1200,"template":{"data":{"histogram2dcontour":[{"type":"hi

##### *CotA Figures*

In [91]:
#CotA MT Grade Summary Table
cotasumjson = cotasumfig.to_json()

cotasumhtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {cotasumjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(cotasumhtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"cells":{"fill":{"color":[["#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF"],["#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1CC","#FFFFFF","#FFF1

In [92]:
#CotA Interventions Summary Table
cotainterjson = cotainterfig.to_json()

cotainterhtml = f"""
<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {cotainterjson};

function renderPlot() {{
    var container = document.getElementById('plotly-div');
    if (container) {{
        Plotly.newPlot(container, fig.data, fig.layout, {{responsive: true}});
    }}
}}

// Initial render
setTimeout(renderPlot, 300);

// Re-render if Power BI wipes it
setInterval(function() {{
    var container = document.getElementById('plotly-div');
    if (container && container.children.length === 0) {{
        renderPlot();
    }}
}}, 2000);
</script>
"""

print(cotainterhtml)


<div id="plotly-div"></div>

<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

<script>
var fig = {"data":[{"cells":{"fill":{"color":[["#FFFFFF","#FFF1CC","#EFAB00"],["#FFF1CC","#FFF1CC","#EFAB00"],["#FFFFFF","#FFF1CC","#EFAB00"]]},"font":{"color":"black"},"line":{"color":"black"},"values":[[1,2," "],["CotA Major, FR or SO, CotA Course, midterm D+ or Below, still registered","CotA Major, JR or SR, CotA Course, midterm F, still registered","Total"],[[268],[40],[308]]]},"header":{"fill":{"color":"#EFAB00"},"font":{"color":"white","weight":"bold"},"line":{"color":"black"},"values":["Group #","CotA Midterm Intervention - Student Criteria","# of Instances"]},"visible":true,"type":"table"}],"layout":{"height":400,"title":{"font":{"weight":"bold"},"text":"\u003cb\u003eSpring 2026 CotA Advisor Intervention Table","x":0.5,"xanchor":"center"},"width":1200,"template":{"data":{"histogram2dcontour":[{"type":"histogram2dcontour","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":